# A Global Mini Integrated Assessment Model: Energy Transitions under a Carbon Budget

**ENGG562/ENV462 · Climate Change Governance: Science, Data & Models · LUMS · Fall 2026**

**Module 2: Scenarios & Integrated Assessment** · Coding exercise 05 · Mini integrated assessment model series, notebook 3 of 5 · Requires coding exercise 04

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/05-global-mini-iam.ipynb)

Coding exercise 04 built a MESSAGE-type model of the fictional country Westeros. This notebook applies the same formulation, with the same commands, to a stylised model of the global energy system from 2020 to 2100. The model solves in about one second and addresses the main questions studied with the IAM scenarios of the IPCC Sixth Assessment Report (coding exercise 02): how fast emissions must fall to stay within a carbon budget, which technologies carry the transition, how high the carbon price must be, and what role negative emissions play.


**Data.** The input data are rounded, illustrative values of a plausible order of magnitude (see Part 1). The model is a teaching tool for understanding how integrated assessment models work. Its results are not projections and must not be cited as such. The AR6 scenario ensemble of coding exercise 02 is the appropriate source for projections.

**Running the notebook.** Open the notebook in Google Colab with the button above ([direct link](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/05-global-mini-iam.ipynb)) and select *File → Save a copy in Drive* before making any changes; work saved without this step is lost when the session ends. Run the cells in order, starting with Part 0. Colab resets the session after a period of inactivity; after a reset, run Part 0 again before continuing.

**Conventions.** Cells headed *Exercise* contain gaps, marked `...` or `___`, that must be completed. The self-check cell after an exercise prints PASS or FAIL for each check. Prompts headed *Prediction* or *Question* require a written answer in the cell that follows.

## Learning objectives

On completing this notebook, students should be able to:

1. Describe the reference energy system of a global integrated assessment model and relate each element to its Westeros counterpart.
2. Check the base-year calibration of a model against observed data.
3. Translate a remaining carbon budget from the IPCC into a model constraint, and estimate the warming of a pathway with the transient climate response to cumulative emissions (TCRE).
4. Compare carbon-budget and carbon-tax scenarios in terms of emissions, technologies, carbon prices, and costs.
5. Run sensitivity analyses on contested assumptions (availability of carbon capture, bioenergy, nuclear power, the pace of solar expansion) and explain their effect.



## In-class session

The class works in 10 groups of 4–5 students. Each group tests one contested assumption of the model (Part 6.4), and each student within a group tests a personal level of that assumption, generated from the student's roll number. Results therefore differ between students, and the instructor can reproduce each result from the roll number.

| Groups | Assumption tested in Part 6.4 | What it represents |
|---|---|---|
| 1 and 6 | Biomass potential | How much land can be given to energy crops without harming food production and ecosystems |
| 2 and 7 | Wind and solar share | How much variable renewable electricity the grid can absorb without large-scale storage |
| 3 and 8 | Nuclear capacity | Whether new nuclear power is politically and financially possible |
| 4 and 9 | Delayed action | Whether the world keeps burning fossil fuels at close to baseline levels until 2030 |
| 5 and 10 | Discount rate | How much the model values costs in the future relative to costs today |

Each student also receives a personal cap on bioenergy with carbon capture and storage (BECCS) and a personal carbon budget.

**Session plan**

| Time | Activity | Where |
|---|---|---|
| 0–10 min | Prediction: the smallest carbon budget the default model can meet | Part 6.4, Step 0 |
| 10–25 min | Individual: write a bisection search for the feasibility frontier | Exercise 7(a) |
| 25–45 min | Individual: four frontiers, the interaction effect, and the personal budget scenario | Exercise 7(b)–(c) |
| 45–60 min | Group: combine the members' results and run a controlled sweep | Exercise 7(d) |
| 60–75 min | Post to GitHub Discussions, then reply to one student from another group | Exercise 7(e) |

**Deliverables.** (1) One group post in the Discussions thread *Exercise 05: Feasibility frontier results*, with the group figure and the group's answer; (2) one individual reply under the group post, with the personal figure and answers generated by the notebook; (3) one peer reply to a student from the other group that tests the same assumption. Roll numbers must not appear in any post; each case is identified by its six-character case code.

Parts 1 to 5 must be completed before the session.

---
# Part 0 · Setup

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyomo highspy
    !wget --quiet --timestamping https://raw.githubusercontent.com/LUMS-WIT/climate-futures/main/content-src/coding/miniiam.py
else:
    # Local run: miniiam.py sits next to the student copies, one folder above the trainer copies
    found = [d for d in (".", "..", "content-src/coding") if os.path.exists(os.path.join(d, "miniiam.py"))]
    if found:
        sys.path.insert(0, os.path.abspath(found[0]))
    else:
        print("miniiam.py not found from", os.getcwd(), "- place it next to this notebook.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import miniiam as mi
from miniiam import make_df

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)

GWA_TO_PWH = 8.76 / 1000   # 1 GWa = 8.76 TWh
GWA_TO_EJ = 0.031536       # 1 GWa = 0.031536 EJ

In [ ]:
def check(description, condition):
    """Self-check helper: prints a tick or a cross. It never stops the notebook."""
    try:
        passed = bool(condition())
    except Exception:
        passed = False
    print(("PASS  " if passed else "FAIL  ") + description)

---
# Part 1 · The global reference energy system

The world model has the same four levels as Westeros, with one more at the start of the chain: **primary** energy (fuels as extracted), **secondary** energy (electricity from power plants, refined oil products), **final** energy (delivered to users), and **useful** energy (the service). Three useful-energy demands drive the model:

- `spec`: **specific electricity uses** that only electricity can supply (appliances, motors, lighting, information technology). This is the "light" of Westeros at a global scale.
- `therm`: **thermal energy** for industry and buildings (process heat, space and water heating, cooking).
- `transport`: **transport services**, expressed as useful energy delivered by vehicles.

```
 PRIMARY              SECONDARY                    FINAL                      USEFUL
 coal_extr ─ coal ──┬─ coal_ppl, coal_ccs ─┐
 gas_extr ── gas ───┼─ gas_ppl, gas_ccs ───┤                    ┌─ elec_spec ─────────▶ spec
 oil_extr ─ crude ─ oil_ref ─ oil ─────────┼─────────────┐      ├─ elec_i, h2_i ──────▶ therm  ◀─ coal_i, gas_i, oil_i, bio_i
 bio_extr ── biomass┼─ bio_ppl, bio_ccs ───┤ electricity ─ elec_t_d ─ electricity ─┤
                    nuc_ppl, hydro_ppl ────┤                    ├─ elec_trp, h2_trp ──▶ transport ◀─ oil_trp, gas_trp, bio_trp
                    wind_ppl, solar_pv ────┘                    └─ h2_elec ─ hydrogen
```

The model is built by the function `mi.build_world()`, which uses only the commands of notebook 04 (`add_horizon`, `add_set`, `make_df`, `add_par`). Its data are stored in readable tables in the module.

### The technologies in the model

Each name in the diagram represents a class of equipment. The tables below describe what each technology represents, with an example where one helps, and what it converts. **Efficiency** is the output per unit of input. For power plants it is the share of the fuel's energy that becomes electricity; for end-use technologies it is the useful energy delivered per unit of final energy, so it includes the losses of engines, boilers, and motors. Technologies without an input fuel (nuclear, hydro, wind, solar) are counted directly by their electricity output.

**Primary energy and fuel processing**

| Name | Technology | What it represents | Converts | Efficiency |
|---|---|---|---|---|
| `coal_extr` | Coal supply | Mining of hard coal and lignite, and coal imports (for example, the Thar lignite field in Sindh) | resource → coal | — |
| `gas_extr` | Natural gas supply | Gas fields and imports by pipeline or as liquefied natural gas (LNG) | resource → gas | — |
| `oil_extr` | Crude oil supply | Oil fields and imports of crude oil | resource → crude oil | — |
| `bio_extr` | Biomass supply | Energy crops, wood, and agricultural residues such as bagasse and rice straw, collected and delivered as fuel | resource → biomass | — |
| `oil_ref` | Oil refinery | Conversion of crude oil into petrol, diesel, kerosene, and furnace oil | crude oil → oil products | 0.90 |

**Electricity generation**

| Name | Technology | What it represents | Converts | Efficiency |
|---|---|---|---|---|
| `coal_ppl` | Coal power plant | Steam power plant burning pulverised coal; the largest single source of electricity in the world in 2020 | coal → electricity | 0.37 |
| `coal_ccs` | Coal power with carbon capture and storage (CCS) | Coal plant that captures about 90% of the CO₂ in its flue gas and stores it underground; very few operate (for example, Boundary Dam in Canada, since 2014). Capture uses energy, hence the lower efficiency | coal → electricity | 0.30 |
| `gas_ppl` | Gas power plant | Combined-cycle gas turbine (CCGT), in which a gas turbine and a steam turbine share the same fuel | gas → electricity | 0.52 |
| `gas_ccs` | Gas power with CCS | Combined-cycle plant with capture of the CO₂ in its exhaust; at the demonstration stage | gas → electricity | 0.45 |
| `nuc_ppl` | Nuclear power plant | Fission reactors (for example, the Chashma units and the K-2 and K-3 units near Karachi) | → electricity | — |
| `hydro_ppl` | Hydropower | Storage dams and run-of-river plants (for example, Tarbela on the Indus); limited by a global potential | → electricity | — |
| `wind_ppl` | Wind power | Onshore wind turbines (for example, the Jhimpir–Gharo wind corridor in Sindh) | → electricity | — |
| `solar_pv` | Solar photovoltaics (PV) | Utility-scale solar parks (for example, the Quaid-e-Azam Solar Park in Bahawalpur) and rooftop panels | → electricity | — |
| `bio_ppl` | Biomass power plant | Steam plants burning wood pellets or agricultural residues (for example, bagasse-fired plants at sugar mills) | biomass → electricity | 0.30 |
| `bio_ccs` | Bioenergy with CCS (BECCS) | Biomass power plant with carbon capture. The carbon in the biomass was taken from the atmosphere as the plants grew, so storing it underground gives **net negative** emissions. Operating examples are few and small | biomass → electricity | 0.25 |

**Networks and hydrogen**

| Name | Technology | What it represents | Converts | Efficiency |
|---|---|---|---|---|
| `elec_t_d` | Transmission and distribution | High-voltage lines, substations, and local distribution networks; the model assumes 10% losses, which is below the losses of many national grids, including Pakistan's | electricity (secondary) → electricity (final) | 0.90 |
| `h2_elec` | Electrolyser | Splitting water into hydrogen and oxygen with electricity; produces "green" hydrogen when the electricity is low-carbon | electricity → hydrogen | 0.65 |

**End use: specific electricity, thermal energy, and transport**

| Name | Technology | What it represents | Converts | Efficiency |
|---|---|---|---|---|
| `elec_spec` | Appliances and motors | Lighting, fans, air conditioners, electric motors, and information technology, which only electricity can supply | electricity → `spec` | 1.00 |
| `coal_i` | Coal-fired heat | Industrial boilers and kilns (cement, steel, bricks) burning coal | coal → `therm` | 0.70 |
| `gas_i` | Gas-fired heat | Industrial boilers, domestic water heaters, space heaters, and cooking stoves on piped gas | gas → `therm` | 0.85 |
| `oil_i` | Oil-fired heat | Boilers and furnaces burning furnace oil, diesel, or kerosene | oil products → `therm` | 0.80 |
| `bio_i` | Biomass heat | Cooking and heating with wood, dung, and crop residues, and industrial biomass boilers; the low efficiency reflects traditional stoves | biomass → `therm` | 0.50 |
| `elec_i` | Electric heat | Heat pumps, electric furnaces, and resistance heaters. The efficiency exceeds 1 because a heat pump moves more heat than the electricity it consumes | electricity → `therm` | 1.50 |
| `h2_i` | Hydrogen heat | Hydrogen burners for high-temperature industrial heat, and hydrogen in steelmaking | hydrogen → `therm` | 0.90 |
| `oil_trp` | Petrol and diesel vehicles | Cars, motorcycles, trucks, buses, ships, and aircraft with combustion engines; most of the fuel energy is lost as heat | oil products → `transport` | 0.25 |
| `gas_trp` | Gas vehicles | Vehicles running on compressed natural gas (CNG) or LNG, such as the CNG cars and rickshaws common in Pakistan | gas → `transport` | 0.25 |
| `bio_trp` | Biofuel vehicles | Vehicles running on ethanol or biodiesel; the efficiency includes the losses of converting biomass into liquid fuel | biomass → `transport` | 0.15 |
| `elec_trp` | Electric vehicles | Battery electric cars, two- and three-wheelers, buses, and electric rail | electricity → `transport` | 0.75 |
| `h2_trp` | Fuel-cell vehicles | Trucks and buses that convert hydrogen into electricity on board | hydrogen → `transport` | 0.45 |

The variable cost of an end-use technology (Part 1 data below) stands for the cost of the equipment itself, such as a vehicle or a heat pump, relative to a conventional option. For this reason, electric vehicles are expensive at the start of the horizon and become cheaper over time in the model.

In [ ]:
base = mi.build_world(mi.Scenario(model="World", scenario="baseline"))

tec = pd.DataFrame(mi.WORLD_TECH, index=["input", "input level", "output", "output level", "efficiency"]).T
tec

In [ ]:
cap = pd.DataFrame({t: {"inv_cost 2030 (USD/kW)": mi._interp(d["inv"], 2030),
                        "inv_cost 2050 (USD/kW)": mi._interp(d["inv"], 2050),
                        "fix_cost (USD/kW/yr)": d["fix"], "var_cost (USD/kWa)": d["var"],
                        "capacity factor": d["cf"], "lifetime (yr)": d["life"]}
                    for t, d in mi.WORLD_CAPACITY.items()}).T
cap

**Where the numbers come from.** All values are rounded, illustrative assumptions of a plausible order of magnitude around 2020. Fuel costs are expressed per kWa: 80 USD/kWa for coal corresponds to about 2.5 USD per GJ, and 300 USD/kWa for crude oil to about 9.5 USD per GJ (roughly 55–60 USD per barrel). CO₂ emission factors of fuels are the IPCC 2006 default values (94.6 kg CO₂ per GJ for coal, 56.1 for natural gas, 73.3 for crude oil). CCS plants capture 90% of the CO₂ in their fuel; for biomass with CCS this gives **net negative** emissions, because the carbon in the biomass was taken from the atmosphere as the plants grew.

Three groups of constraints keep the linear model from producing physically implausible results:

- **Growth constraints** (`growth_activity_up`, `initial_activity_up`): for example, solar and wind output can grow by at most 10% per year, plus a start-up allowance.
- **Decline constraints** (`growth_activity_lo`): existing fossil technologies can shrink by at most 4–6% per year, reflecting the slow turnover of capital stock and infrastructure.
- **Potentials** (`bound_activity_up`): biomass supply is limited to about 120 EJ per year and hydropower to about 7,900 TWh per year. A **relation** (`relation_activity`, named `vre_share`) limits wind and solar to 60% of electricity generation, an approximation of the storage and grid flexibility that a model with one time slice per year cannot represent.

In [ ]:
base.par("growth_activity_up").groupby("technology")["value"].first().to_frame("max. annual growth").T

---
# Part 2 · Checking the base year

Before any scenario is interpreted, a modeller checks that the base year resembles the real world. The model is given the activity of each technology in 2020 (`historical_activity`). From it, total electricity generation, final energy, and CO₂ emissions in 2020 can be computed and compared with observed data.

Observed reference values: global electricity generation in 2020 was about 26,800 TWh, of which 7,593 TWh renewable (IEA, *World Energy Outlook 2021*); total final consumption was of the order of 385–400 EJ; and fossil CO₂ emissions, including cement production, were 34.8 Gt (Global Carbon Budget 2021).

### Exercise 1: Base-year emissions

Compute the CO₂ emissions implied by the 2020 activity. Each technology that burns a fossil fuel uses `activity / efficiency` units of fuel, and each unit of fuel emits the factor in `mi.EF` (Mt CO₂ per GWa). Oil products pass through the refinery (`oil_ref`, efficiency 0.9) before use, so their crude-oil use is a further factor 1/0.9 larger. Store the total in Gt as `co2_2020`.

In [ ]:
hist = mi.WORLD_HIST_2020
fuel_use = {"coal": 0.0, "gas": 0.0, "oil": 0.0}
for t, act in hist.items():
    inp, _, _, _, eff = mi.WORLD_TECH[t]
    if inp in ("coal", "gas"):
        fuel_use[inp] += act / eff
    elif inp == "oil":
        fuel_use["oil"] += act / eff / ___

co2_2020 = sum(fuel_use[f] * mi.EF[f] for f in fuel_use) / ___
print(f"Implied CO2 emissions in 2020: {co2_2020:.1f} Gt")

In [ ]:
check("about 30 Gt CO2", lambda: 28 < co2_2020 < 33)

In [ ]:
gen = [t for t, v in mi.WORLD_TECH.items() if v[2] == "electricity" and v[3] == "secondary"]
elec_2020 = sum(mi.WORLD_HIST_2020.get(t, 0) for t in gen) * GWA_TO_PWH * 1000
renew_2020 = sum(mi.WORLD_HIST_2020.get(t, 0) for t in ["hydro_ppl", "wind_ppl", "solar_pv", "bio_ppl"]) * GWA_TO_PWH * 1000
print(f"Electricity generation 2020: {elec_2020:,.0f} TWh, of which renewable {renew_2020:,.0f} TWh")

**Question.** The model's CO₂ emissions in 2020 are lower than the 34.8 Gt reported by the Global Carbon Budget. List at least two sources of CO₂ that the model does not include, and state whether the omission matters for the questions studied in this notebook.

*Answer:* *(write the answer here)*

---
# Part 3 · The baseline scenario

The baseline has no climate policy. Demand for useful energy grows along an SSP2-like path ("middle of the road", Lecture 8): specific electricity uses grow by 2.5% per year to 2050, transport by 2%, and thermal energy by 1%.

In [ ]:
base.solve()
print(f"Total discounted system cost: {base.var('OBJ') / 1e6:.1f} trillion USD")

In [ ]:
COLORS = {"coal_ppl": "#4d4d4d", "coal_ccs": "#8c8c8c", "gas_ppl": "#e08214", "gas_ccs": "#fdb863",
          "nuc_ppl": "#8073ac", "hydro_ppl": "#4393c3", "wind_ppl": "#5aae61", "solar_pv": "#fee08b",
          "bio_ppl": "#a6611a", "bio_ccs": "#dfc27d"}


def plot_power(s, ax=None, title=None):
    act = mi.timeseries(s, "ACT")[gen] * GWA_TO_PWH
    ax = ax or plt.gca()
    act.plot.area(ax=ax, color=[COLORS[t] for t in gen], linewidth=0, legend=False)
    ax.set(title=title or f"Electricity generation: {s.scenario}", ylabel="PWh per year", xlabel="")
    return ax


def emissions(s):
    return s.var("EMISS").set_index("year")["lvl"] / 1000   # Gt CO2 per year


fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_power(base, axes[0])
axes[0].legend(loc="upper left", fontsize=8, ncol=2)
emissions(base).plot(ax=axes[1], marker="o", color="black", title="CO2 emissions (Gt per year)")
axes[1].axhline(0, color="grey", lw=0.5)
plt.tight_layout()
plt.show()

The end-use technologies show how demand is met in each sector.

In [ ]:
END_USE = {"therm": ["coal_i", "gas_i", "oil_i", "bio_i", "elec_i", "h2_i"],
           "transport": ["oil_trp", "gas_trp", "bio_trp", "elec_trp", "h2_trp"]}

def plot_end_use(s):
    act = mi.timeseries(s, "ACT")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (sector, techs) in zip(axes, END_USE.items()):
        act[techs].plot.area(ax=ax, linewidth=0, title=f"{sector}: useful energy by technology (GWa)")
        ax.legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.show()

plot_end_use(base)

**Question.** Describe the baseline in three sentences: which energy source dominates electricity, how transport changes, and what happens to emissions. Then identify one feature of the result that is a consequence of the model being **linear** rather than a realistic expectation.

*Answer:* *(write the answer here)*

### Exercise 2: What the constraints do

Remove all growth and decline constraints from a clone of the baseline (`remove_par("growth_activity_up")`, and the same for `initial_activity_up` and `growth_activity_lo`), solve it as `free`, and plot its end-use technologies with `plot_end_use(free)`. Compute `coal_i_2030`, the activity of `coal_i` in 2030 in the unconstrained run.

In [ ]:
free = base.clone(scenario="no dynamic constraints")
for name in [___, ___, ___]:
    free.remove_par(name)
free.solve()
plot_end_use(free)
coal_i_2030 = mi.timeseries(free, "ACT").loc[2030, ___]
print(f"coal for heat in 2030: {coal_i_2030:,.0f} GWa (baseline: {mi.timeseries(base, 'ACT').loc[2030, 'coal_i']:,.0f})")

In [ ]:
check("without dynamic constraints, coal takes over thermal demand in 2030",
      lambda: coal_i_2030 > 2 * mi.timeseries(base, "ACT").loc[2030, "coal_i"])

**Question.** Without dynamic constraints, the model replaces almost all gas and oil boilers with coal within ten years. Why is this unrealistic, and what does the exercise reveal about how results of large IAMs depend on such constraints?

*Answer:* *(write the answer here)*

---
# Part 4 · Carbon budgets for 2 °C and 1.5 °C

Global warming is roughly proportional to cumulative CO₂ emissions. The constant of proportionality is the **transient climate response to cumulative emissions (TCRE)**, which the IPCC (AR6 WGI) assesses at 0.45 °C per 1,000 Gt CO₂ (likely range 0.27–0.63). The IPCC derives from it **remaining carbon budgets** from the start of 2020:

| Target | Remaining budget from 2020 |
|---|---|
| 1.5 °C with 50% probability | 500 Gt CO₂ |
| 2 °C with 67% probability | 1,150 Gt CO₂ |

These budgets already account for the expected warming from non-CO₂ gases. As in notebook 04, a `bound_emission` with `type_year="cumulative"` limits the **average** annual emissions over the model horizon, so a budget is entered as its total divided by the number of years in the horizon (2021–2100, 80 years).

In [ ]:
horizon_years = sum(base.par("duration_period").set_index("year").loc[base.model_years(), "value"])


def budget_scenario(budget_gt, name, start=base):
    s = start.clone(scenario=name)
    s.add_par("bound_emission", make_df("bound_emission", node="World", type_emission="GHG", type_tec="all",
                                        type_year="cumulative", value=budget_gt * 1000 / horizon_years,
                                        unit="MtCO2"))
    s.solve()
    return s


print("horizon length:", horizon_years, "years")
b2c = budget_scenario(1150, "2C (1150 Gt)")

**Prediction (before running the next cell).** How will the 2 °C budget change the electricity mix compared with the baseline, and roughly when will global CO₂ emissions reach net zero?

*Answer:* *(write the answer here)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
plot_power(base, axes[0])
plot_power(b2c, axes[1])
axes[1].legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### Exercise 3: The 1.5 °C budget

Create the 1.5 °C scenario `b15` with the function `budget_scenario`. Then build a table `summary` with one row per scenario (baseline, 2 °C, 1.5 °C) and the columns: emissions in 2030 and 2050 (Gt), the first year with net-zero or negative emissions, cumulative emissions 2021–2100 (Gt), and peak warming (°C, from `mi.warming`).

In [ ]:
b15 = budget_scenario(___, "1.5C (500 Gt)")

def summarise(s):
    e = emissions(s)
    net_zero = e[e <= 0].index.min()
    return {"CO2 2030 (Gt)": e[2030], "CO2 2050 (Gt)": e[___],
            "net zero": net_zero,
            "cumulative 2021-2100 (Gt)": mi.cumulative_co2(s).iloc[-1],
            "peak warming (°C)": mi.warming(s).___()}

summary = pd.DataFrame({s.scenario: summarise(s) for s in [base, b2c, b15]}).T
summary

In [ ]:
check("the 1.5 °C scenario stays within 500 Gt", lambda: summary.loc["1.5C (500 Gt)", "cumulative 2021-2100 (Gt)"] <= 500.5)
check("the 1.5 °C scenario cuts 2050 emissions more than the 2 °C scenario",
      lambda: summary.loc["1.5C (500 Gt)", "CO2 2050 (Gt)"] < summary.loc["2C (1150 Gt)", "CO2 2050 (Gt)"])

`mi.warming()` adds the CO₂-induced warming (TCRE × cumulative emissions since 2020) to the observed warming of about 1.1 °C. It ignores non-CO₂ gases and is therefore a lower estimate; it is useful for comparing scenarios, not for stating whether a target is met.

For comparison, in the AR6 scenarios that limit warming to 1.5 °C with no or limited overshoot (category C1), global net CO₂ emissions reach zero in the early 2050s; in the scenarios that limit warming to 2 °C with 67% probability (C3), in the early 2070s (IPCC AR6 WGIII, Summary for Policymakers, C.1).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for s, c in [(base, "black"), (b2c, "tab:blue"), (b15, "tab:green")]:
    emissions(s).plot(ax=axes[0], marker="o", color=c, label=s.scenario)
    p = s.var("PRICE_EMISSION")
    if len(p):
        p.set_index("year")["lvl"].plot(ax=axes[1], marker="o", color=c, label=s.scenario, logy=True)
axes[0].axhline(0, color="grey", lw=0.5)
axes[0].set(title="CO2 emissions (Gt per year)")
axes[1].set(title="Carbon price (USD per t CO2, log scale)")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

**Question.** Compare the timing of net zero in the model's 1.5 °C scenario with the AR6 C1 scenarios, and compare the model's carbon price in 2030 with the price that the shadow price of BECCS implies in 2100. Explain the connection between the two observations.

*Answer:* *(write the answer here)*

---
# Part 5 · A carbon tax

Instead of a budget, a government can set a carbon price. MESSAGE implements it with `tax_emission`. A common design is a price that starts at a moderate level and rises at a fixed rate.

In [ ]:
years = base.model_years()
tax_path = [30 * 1.05 ** (y - 2030) for y in years]   # 30 USD/t in 2030, rising by 5% per year

tax = base.clone(scenario="tax 30 USD/t +5%/yr")
tax.add_par("tax_emission", make_df("tax_emission", node="World", type_emission="GHG", type_tec="all",
                                    type_year=years, value=tax_path, unit="USD/tCO2"))
tax.solve()
pd.Series(tax_path, index=years, name="tax (USD/t)").round(0).to_frame().T

In [ ]:
summary_tax = pd.DataFrame({s.scenario: summarise(s) for s in [base, b2c, b15, tax]}).T
summary_tax

**Question.** Compare the cumulative emissions of the tax scenario with the two budget scenarios. A government that sets a carbon tax does not know in advance which budget it will achieve. What does this imply for the choice between a tax and a cap (notebook 03, Weitzman 1974) when the costs of technologies are uncertain?

*Answer:* *(write the answer here)*

### Cost of the transition

The objective value measures the total discounted cost of the energy system. Comparing it across scenarios gives the **mitigation cost**, which must exclude tax payments (they are transfers). `COST_NODAL_NET` reports the annual system cost without taxes.

In [ ]:
def system_cost(s):
    return s.var("COST_NODAL_NET").set_index("year")["lvl"] / 1e3   # billion USD per year

cost = pd.DataFrame({s.scenario: system_cost(s) for s in [base, b2c, b15, tax]})
(cost.sub(cost[base.scenario], axis=0)).drop(columns=base.scenario).round(0)

**Question.** The table shows the additional annual energy-system cost of each scenario relative to the baseline, in billion USD. Why is this not the same as the cost of climate policy to the economy, and which model in this series addresses the difference?

*Answer:* *(write the answer here)*

---
# Part 6 · In-class activity: contested assumptions

Integrated assessment results depend on assumptions about which experts disagree. In this activity, groups test how the 2 °C scenario changes when one such assumption changes. The cell below provides a helper that applies a change to a clone of the baseline and then imposes the 2 °C budget.

In [ ]:
def variant(name, change, budget_gt=1150):
    s = base.clone(scenario=name)
    change(s)
    try:
        return budget_scenario(budget_gt, name, start=s)
    except RuntimeError as err:
        print(f"{name}: no feasible solution. {err}")
        return None


def zero_bound(techs):
    def change(s):
        for t in techs:
            s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology=t,
                                                   year_act=years, mode="standard", time="year",
                                                   value=0.0, unit="GWa"))
    return change

## 6.1 Core task (all groups)

### Exercise 4: A world without carbon capture

CCS is not yet deployed at scale, and whether it will be is contested. Create the variant `no_ccs`, in which `coal_ccs`, `gas_ccs`, and `bio_ccs` cannot be used, using `variant` and `zero_bound`. Add it to the comparison table and plot its electricity mix.

In [ ]:
no_ccs = variant("2C without CCS", zero_bound([___, ___, ___]))
comparison = pd.DataFrame({s.scenario: summarise(s) for s in [b2c, no_ccs]}).T
comparison["carbon price 2030"] = [s.var("PRICE_EMISSION").set_index("year").loc[2030, "lvl"] for s in [b2c, no_ccs]]
print(comparison)
plot_power(no_ccs)
plt.show()

In [ ]:
check("without CCS, emissions in 2050 are lower", lambda: emissions(no_ccs)[2050] < emissions(b2c)[2050])
check("without CCS, the 2030 carbon price is higher",
      lambda: comparison.loc["2C without CCS", "carbon price 2030"] > comparison.loc["2C (1150 Gt)", "carbon price 2030"])

**Question.** Explain why removing CCS **raises** the near-term carbon price and **lowers** emissions in 2050, even though the budget is unchanged.

*Answer:* *(write the answer here)*

## 6.2 ENGG track: the power sector

### Exercise 5: Nuclear phase-out and the pace of solar

**(a)** Create a variant `no_new_nuclear` in which nuclear activity cannot exceed its 2020 level of 310 GWa (use `bound_activity_up` with value 310 in all years). **(b)** Create a variant `fast_solar` in which the maximum growth rate of `solar_pv` and `wind_ppl` is raised from 10% to 15% per year. Compare both with `b2c` in terms of the carbon price in 2030 and 2050, and the share of solar in electricity in 2050.

In [ ]:
def cap_nuclear(s):
    s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="nuc_ppl",
                                           year_act=years, mode="standard", time="year", value=___, unit="GWa"))

def faster_vre(s):
    for t in ["solar_pv", "wind_ppl"]:
        s.add_par("growth_activity_up", make_df("growth_activity_up", node_loc="World", technology=t,
                                                year_act=years, time="year", value=___, unit="-"))

no_new_nuclear = variant("2C nuclear capped", cap_nuclear)
fast_solar = variant("2C fast VRE", faster_vre)

def solar_share(s, y=2050):
    a = mi.timeseries(s, "ACT").loc[y, gen]
    return a["solar_pv"] / a.sum()

engg = pd.DataFrame({s.scenario: {"price 2030": s.var("PRICE_EMISSION").set_index("year").loc[2030, "lvl"],
                                  "price 2050": s.var("PRICE_EMISSION").set_index("year").loc[2050, "lvl"],
                                  "solar share 2050": solar_share(s)}
                     for s in [b2c, no_new_nuclear, fast_solar]}).T
engg.round(2)

In [ ]:
check("capping nuclear raises the carbon price", lambda: engg.loc["2C nuclear capped", "price 2050"] >= engg.loc["2C (1150 Gt)", "price 2050"])
check("faster VRE growth changes the carbon price by less than 2%",
      lambda: abs(engg.loc["2C fast VRE", "price 2050"] / engg.loc["2C (1150 Gt)", "price 2050"] - 1) < 0.02)

**Question.** Capping nuclear power raises the carbon price, but faster growth of solar and wind hardly changes it. Use the constraints described in Part 1 to explain why, and suggest which additional change would let faster solar growth lower the cost of the 2 °C target.

*Answer:* *(write the answer here)*

## 6.3 ENV track: bioenergy and land

### Exercise 6: How much biomass is sustainable?

Global bioenergy potentials are highly uncertain because biomass competes with food production and ecosystems for land. Run the 2 °C scenario with a biomass limit of 1,600 GWa per year (about 50 EJ, roughly the current level) instead of 3,800 GWa. Name the variant `low_bio`, and report its cumulative negative emissions from `bio_ccs` (Gt), its carbon price in 2050, and its net-zero year, compared with `b2c`.

In [ ]:
def less_biomass(s):
    s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="bio_extr",
                                           year_act=years, mode="standard", time="year", value=___, unit="GWa"))

low_bio = variant("2C low biomass", less_biomass)

def beccs_removals(s):
    ef = -mi.CAPTURE * mi.EF["biomass"] / 0.25           # Mt CO2 removed per GWa of electricity
    act = mi.timeseries(s, "ACT")["bio_ccs"]
    return -(act * ef * 10).sum() / 1000                 # Gt over the horizon (10-year periods)

env = pd.DataFrame({s.scenario: {"BECCS removals (Gt)": beccs_removals(s),
                                 "price 2050": s.var("PRICE_EMISSION").set_index("year").loc[2050, "lvl"],
                                 "net zero": summarise(s)["net zero"]}
                    for s in [b2c, low_bio]}).T
env

In [ ]:
check("less biomass raises the carbon price", lambda: env.loc["2C low biomass", "price 2050"] >= env.loc["2C (1150 Gt)", "price 2050"])

**Question.** This model treats biomass as a single commodity with a fixed cost and no land. List three consequences of large-scale bioenergy that a coupled land-use model (such as GLOBIOM, which is linked to MESSAGE in MESSAGEix-GLOBIOM) would capture and this model ignores.

*Answer:* *(write the answer here)*

## 6.4 Individual task: the feasibility frontier

The **feasibility frontier** is the smallest cumulative CO₂ budget (2021–2100) for which the model still finds a solution. No budget below the frontier can be met with the technologies and constraints in the model, at any cost. The frontier therefore shows what the model must assume for a given target to be met.

### Step 0: Prediction

Before running any code, write down the smallest budget that the **default** model can meet. The 1.5 °C budget is 500 Gt and the 2 °C budget is 1,150 Gt. The prediction is collected in class before Exercise 7(a).

*Prediction:* *(write the number here)*

### Personal case

Enter the roll number and the group number. The function `my_case` turns them into a personal case: a carbon budget, a BECCS cap, and a level of the group's assumption. The same inputs always give the same case.

In [ ]:
import hashlib, io, contextlib

ROLL_NUMBER = "________"   # LUMS roll number, for example "26100123"
GROUP = 0                  # group number, 1 to 10

ASSUMPTIONS = ["biomass potential", "wind and solar share", "nuclear capacity", "delayed action", "discount rate"]
LEVELS = {"biomass potential": (1200, 3000),     # GWa per year of biomass (default 3,800)
          "wind and solar share": (0.30, 0.55),  # maximum share in electricity (default 0.60)
          "nuclear capacity": (150, 350),        # GWa per year of nuclear output (2020: 310)
          "delayed action": (0.80, 1.00),        # share of baseline fossil use kept in 2030
          "discount rate": (0.02, 0.08)}         # per year (default 0.05)
DIGITS = {"biomass potential": -1, "wind and solar share": 2, "nuclear capacity": -1,
          "delayed action": 2, "discount rate": 3}


def my_case(roll, group):
    """Personal case derived from the roll number; the assumption is set by the group."""
    h = hashlib.sha256(("ENGG562-ex05-" + str(roll).strip()).encode()).digest()
    u = [int.from_bytes(h[2 * i:2 * i + 2], "big") / 65535 for i in range(3)]
    name = ASSUMPTIONS[(group - 1) % 5]
    lo, hi = LEVELS[name]
    return {"case code": h.hex()[:6].upper(), "group": group,
            "budget (Gt)": int(round(400 + 700 * u[0])),
            "BECCS cap (GWa)": int(round(100 + 500 * u[1], -1)),
            "assumption": name, "level": round(lo + (hi - lo) * u[2], DIGITS[name])}


case = my_case(ROLL_NUMBER, GROUP)
pd.Series(case).to_frame("my case")

In [ ]:
check("roll number and group entered", lambda: ROLL_NUMBER.isdigit() and 1 <= GROUP <= 10)

The next cell provides the changes that the personal case applies to the model, and a function `feasible` that reports whether a budget can be met under a change. It suppresses the solver messages, because the search in Exercise 7(a) solves the model many times.

In [ ]:
def no_change(s):
    pass


def cap_beccs(value):
    def change(s):
        s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="bio_ccs",
                                               year_act=years, mode="standard", time="year", value=value, unit="GWa"))
    return change


def assumption(name, level):
    def change(s):
        if name == "biomass potential":
            s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="bio_extr",
                                                   year_act=years, mode="standard", time="year", value=level, unit="GWa"))
        elif name == "wind and solar share":
            for t in gen:
                coef = (1 - level) if t in ("wind_ppl", "solar_pv") else -level
                s.add_par("relation_activity", make_df("relation_activity", relation="vre_share", node_rel="World",
                                                       year_rel=years, node_loc="World", technology=t, year_act=years,
                                                       mode="standard", value=coef, unit="-"))
        elif name == "nuclear capacity":
            s.add_par("bound_activity_up", make_df("bound_activity_up", node_loc="World", technology="nuc_ppl",
                                                   year_act=years, mode="standard", time="year", value=level, unit="GWa"))
        elif name == "delayed action":
            act_2030 = mi.timeseries(base, "ACT").loc[2030]
            for t in ["coal_ppl", "gas_ppl", "coal_i", "gas_i", "oil_i", "oil_trp"]:
                s.add_par("bound_activity_lo", make_df("bound_activity_lo", node_loc="World", technology=t,
                                                       year_act=2030, mode="standard", time="year",
                                                       value=level * act_2030[t], unit="GWa"))
        elif name == "discount rate":
            s.remove_par("interestrate")
            s.add_par("interestrate", make_df("interestrate", year=years, value=level, unit="-"))
    return change


def combine(*changes):
    def change(s):
        for c in changes:
            c(s)
    return change


def feasible(change, budget_gt):
    """True if the budget scenario under `change` has a solution."""
    with contextlib.redirect_stdout(io.StringIO()):
        return variant("test", change, budget_gt) is not None


print("default model, 1.5 °C budget feasible:", feasible(no_change, 500))

### Exercise 7(a): A frontier finder

Write the function `min_budget`. It must find, by **bisection**, the smallest budget that is still feasible under `change`, to within `tol` Gt. Bisection keeps an interval `[lo, hi]` in which `lo` is infeasible and `hi` is feasible, tests the midpoint, and replaces whichever end the midpoint agrees with, until the interval is shorter than `tol`. The function returns `hi`, or `None` if even the starting `hi` is infeasible. The search starts at `lo = -1000`, because a cumulative budget can be negative: a model with large carbon removals can, in principle, remove more CO₂ over the century than it emits. Feasibility is monotone in the budget: if a budget can be met, every larger budget can be met as well.

No code template is provided. Each solve takes less than one second, and the search requires about ten solves.

In [ ]:
def min_budget(change, lo=-1000, hi=2000, tol=5):
    """Smallest feasible cumulative budget (Gt CO2, 2021-2100) under `change`, to within `tol` Gt."""
    ...


F_default = min_budget(no_change)
print(f"Feasibility frontier of the default model: {F_default:.0f} Gt")

In [ ]:
check("the frontier is feasible", lambda: feasible(no_change, F_default))
check("5 Gt below the frontier is infeasible", lambda: not feasible(no_change, F_default - 5))

**Question.** Compare the frontier with the prediction of Step 0. Which technology allows the default model to meet such a small budget, and in which decades does it do so? (The electricity mix of Part 4 helps.)

*Answer:* *(write the answer here)*

### Exercise 7(b): Four frontiers and their interaction

Compute the frontier under four settings: the default model, the personal BECCS cap alone, the group's assumption alone at the personal level, and both together. The **interaction** measures whether the two assumptions reinforce each other:

$$\text{interaction} = F_\text{both} - F_\text{BECCS} - F_\text{assumption} + F_\text{default}$$

It is zero if the two effects simply add up, positive if together they tighten the frontier by more than the sum of their separate effects, and negative if by less. Each frontier is known only to within 5 Gt, so an interaction smaller than about 10 Gt in absolute value cannot be distinguished from zero.

In [ ]:
my_beccs = cap_beccs(case["BECCS cap (GWa)"])
my_assumption = assumption(case["assumption"], case["level"])

frontiers = pd.Series({"default": F_default,
                       "BECCS cap only": min_budget(___),
                       "assumption only": min_budget(___),
                       "both": min_budget(combine(___, ___))}, name="frontier (Gt)")
interaction = frontiers["both"] - frontiers["BECCS cap only"] - frontiers["assumption only"] + frontiers["default"]
print(frontiers.round(0))
print(f"interaction: {interaction:+.0f} Gt")

In [ ]:
check("each constraint can only raise the frontier",
      lambda: frontiers["both"] >= max(frontiers["BECCS cap only"], frontiers["assumption only"]) - 5
      and frontiers.drop("default").min() >= frontiers["default"] - 5)

### Exercise 7(c): The personal budget

Solve the model at the personal budget twice: once with default assumptions (`reference`) and once with both personal assumptions (`mine`). If `mine` is infeasible, `variant` returns `None`; an infeasible case is a valid result and is reported as such. The next cell then draws the personal figure and saves it as a PNG file for the Discussions post. The figure title carries the case code.

In [ ]:
B = case["budget (Gt)"]
reference = variant(f"default model, {B} Gt", ___, B)
mine = variant(f"case {case['case code']}, {B} Gt", combine(my_beccs, my_assumption), B)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
frontiers.plot.bar(ax=axes[0], color=["grey", "tab:brown", "tab:blue", "black"], rot=0)
axes[0].axhline(B, color="tab:red", ls="--", label=f"personal budget ({B} Gt)")
axes[0].set(title="Feasibility frontier (Gt CO2, 2021-2100)")
axes[0].legend(fontsize=8)
for s, c in [(reference, "grey"), (mine, "black")]:
    if s is not None:
        emissions(s).plot(ax=axes[1], marker="o", color=c, label=s.scenario)
        s.var("PRICE_EMISSION").set_index("year")["lvl"].plot(ax=axes[2], marker="o", color=c,
                                                             label=s.scenario, logy=True)
if mine is None:
    axes[1].text(0.5, 0.5, "personal case infeasible", transform=axes[1].transAxes, ha="center", color="tab:red")
axes[1].axhline(0, color="grey", lw=0.5)
axes[1].set(title="CO2 emissions (Gt per year)", xlabel="")
axes[2].set(title="Carbon price (USD per t CO2, log scale)", xlabel="")
for ax in axes[1:]:
    ax.legend(fontsize=8)
fig.suptitle(f"Exercise 05 · case {case['case code']} · group {GROUP}: BECCS cap {case['BECCS cap (GWa)']} GWa, "
             f"{case['assumption']} = {case['level']}")
plt.tight_layout()
figfile = f"ex05_case_{case['case code']}.png"
fig.savefig(figfile, dpi=150, bbox_inches="tight")
plt.show()

**Questions for the individual post.**

1. **Interaction.** State whether the two personal assumptions together tighten the frontier by more than, less than, or exactly the sum of their separate effects, and explain the mechanism with reference to the technologies that the model uses. Students in groups 5 and 10 should also explain why the discount rate leaves the frontier unchanged, and why this does not make the discount rate irrelevant for policy.
2. **Personal budget.** If the personal case is feasible, explain which change in the energy system meets the budget without the full BECCS deployment, and why the carbon price differs from the reference. If it is infeasible, state the largest BECCS cap that is still insufficient, found with `feasible` and a few trial values.
3. **A sentence for a ministry.** Write one sentence, for an official at Pakistan's Ministry of Climate Change who is reading a model-based claim about the 1.5 °C target, stating which of the two assumptions should be questioned first, and why.

*Answers (draft here, then copy into the post):* *(write the answers here)*

### Exercise 7(d): Group step

The members of a group test the same assumption at different levels, but each member also has a different BECCS cap, so the members' frontiers cannot be compared directly. The group therefore does two things:

1. **Collect.** One member enters every member's case code, level, BECCS cap, and frontier with both assumptions into `group_results` (each member reads them out from Exercise 7(b)).
2. **Control.** The same member runs a **sweep**: the frontier at five levels of the group's assumption, with the BECCS cap held fixed at 300 GWa for everyone. Comparing the two panels of the group figure separates the effect of the assumption from the effect of the BECCS cap.

In [ ]:
group_results = pd.DataFrame([
    (case["case code"], case["level"], case["BECCS cap (GWa)"], frontiers["both"]),
    # ("A1B2C3", 0.42, 300, 410),   # one row per group member
], columns=["case code", "level", "BECCS cap (GWa)", "frontier (Gt)"])

lo, hi = LEVELS[case["assumption"]]
sweep = pd.Series({round(v, DIGITS[case["assumption"]]):
                   min_budget(combine(cap_beccs(300), assumption(case["assumption"], v)))
                   for v in np.linspace(lo, hi, 5)}, name="frontier (Gt)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
pts = axes[0].scatter(group_results["level"], group_results["frontier (Gt)"],
                      c=group_results["BECCS cap (GWa)"], cmap="viridis", vmin=100, vmax=600, s=60)
for _, r in group_results.iterrows():
    axes[0].annotate(r["case code"], (r["level"], r["frontier (Gt)"]), fontsize=7, xytext=(4, 4),
                     textcoords="offset points")
fig.colorbar(pts, ax=axes[0], label="BECCS cap (GWa)")
axes[0].set(title="Group members (BECCS cap varies)", xlabel=case["assumption"], ylabel="frontier (Gt CO2)",
            xlim=(lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo)))
sweep.plot(ax=axes[1], marker="o", color="black",
           title="Sweep (BECCS cap fixed at 300 GWa)", xlabel=case["assumption"])
fig.suptitle(f"Exercise 05 · group {GROUP}: {case['assumption']}")
plt.tight_layout()
groupfile = f"ex05_group_{GROUP:02d}.png"
fig.savefig(groupfile, dpi=150, bbox_inches="tight")
plt.show()

**Question for the group post.** Over the range tested, which matters more for the frontier: the group's assumption or the BECCS cap? Support the answer with both panels of the group figure, and state the level of the assumption, if any, beyond which it stops making a difference.

*Group answer:* *(write the answer here)*

### Exercise 7(e): Post to GitHub Discussions

The next cell prints the individual post with all numbers filled in. Copy it into a reply under the group's post in the Discussions thread *Exercise 05: Feasibility frontier results*, replace the three answer placeholders, and attach the personal figure by dragging the PNG file into the reply box. In Colab, the cell also downloads the figure files.

- **Group post** (one per group, a top-level comment in the thread): the title *Group N: assumption*, the group figure, the `group_results` table, and the group answer.
- **Individual reply** (one per student, under the group post): the text printed below and the personal figure.
- **Peer reply** (one per student): a reply of no more than 100 words to a student from the **other** group that tests the same assumption (group 1 with group 6, 2 with 7, and so on), explaining one difference between the two cases.

The roll number must not appear in any post.

In [ ]:
def results(s):
    if s is None:
        return "infeasible"
    e = emissions(s)
    p = s.var("PRICE_EMISSION").set_index("year")["lvl"]
    nz = e[e <= 0].index.min()
    return (f"CO2 in 2050 {e[2050]:.1f} Gt; net zero {'not reached' if pd.isna(nz) else int(nz)}; "
            f"carbon price {p[2030]:.1f} USD/t in 2030 and {p[2050]:.1f} USD/t in 2050")


post = f"""**Case {case['case code']}** · group {GROUP} · {case['assumption']} = {case['level']} · BECCS cap {case['BECCS cap (GWa)']} GWa · budget {B} Gt

| Frontier | Gt CO2 |
|---|---|
| Default | {frontiers['default']:.0f} |
| BECCS cap only | {frontiers['BECCS cap only']:.0f} |
| Assumption only | {frontiers['assumption only']:.0f} |
| Both | {frontiers['both']:.0f} |
| **Interaction** | **{interaction:+.0f}** |

- Default model at {B} Gt: {results(reference)}
- Personal case at {B} Gt: {results(mine)}

*(drag {figfile} here)*

**1. Interaction.** ...

**2. Personal budget.** ...

**3. Sentence for a ministry.** ...
"""
print(post)

if "google.colab" in sys.modules:
    from google.colab import files
    for f in [figfile, groupfile]:
        files.download(f)

---
# Part 7 · Limitations and references

## What this model leaves out

The model has the structure of MESSAGE, but large IAMs such as MESSAGEix-GLOBIOM differ in ways that matter for the results:

- **Regions and trade:** one global region instead of 11 or 12 world regions linked by trade in fuels and electricity. Pakistan and South Asia are therefore not represented separately.
- **Time slices:** one time step per year instead of seasons and times of day, so the variability of wind and solar is approximated by a share constraint.
- **Vintages:** efficiencies and costs do not vary by the year a plant was built.
- **Land use and non-CO₂ gases:** there is no land model and no methane or nitrous oxide; the climate is represented only by TCRE.
- **Demand:** demand does not respond to prices. Notebook 06 adds price-responsive demand.
- **Data:** all inputs are illustrative.

## References

- IPCC (2021). Summary for Policymakers. In: *Climate Change 2021: The Physical Science Basis.* Contribution of Working Group I to the Sixth Assessment Report of the IPCC. Table SPM.2 (remaining carbon budgets) and Section D.1 (TCRE).
- IPCC (2022). Summary for Policymakers. In: *Climate Change 2022: Mitigation of Climate Change.* Contribution of Working Group III to the Sixth Assessment Report of the IPCC. Section C.1 (timing of net-zero CO₂).
- Anderson, K., and Peters, G. (2016). The trouble with negative emissions. *Science*, 354(6309), 182–183. [doi:10.1126/science.aah4567](https://doi.org/10.1126/science.aah4567)
- Krey, V., Havlík, P., Kishimoto, P. N., Fricko, O., Zilliacus, J., Gidden, M., et al. (2020). *MESSAGEix-GLOBIOM Documentation – 2020 release.* International Institute for Applied Systems Analysis. [doi:10.22022/iacc/03-2021.17115](https://doi.org/10.22022/iacc/03-2021.17115)
- Friedlingstein, P., et al. (2022). Global Carbon Budget 2021. *Earth System Science Data*, 14, 1917–2005. [doi:10.5194/essd-14-1917-2022](https://doi.org/10.5194/essd-14-1917-2022)
- IEA (2021). *World Energy Outlook 2021.* International Energy Agency, Paris.